In [1]:
# =============================================================================
# 1. MONTAJE DE DRIVE E INSTALACIÓN DE DEPENDENCIAS
# =============================================================================
from google.colab import drive
import os
import shutil
import json
import subprocess

# Montar Google Drive
drive.mount('/content/drive')

# Instalar librerías de inferencia cuantizada y validación JSON
!pip install -q -U transformers accelerate bitsandbytes jsonschema

# =============================================================================
# 2. CONFIGURACIÓN DEL BINARIO VALIDADOR EN HASKELL
# =============================================================================
drive_bin_path = "/content/drive/MyDrive/programacion_funcional/haskell_validator/validator"
local_bin_path = "/tmp/validator"

if not os.path.exists(drive_bin_path):
    raise FileNotFoundError(f"No se encontró el binario en: {drive_bin_path}. Verifica la ruta en Drive.")

# Copiar al almacenamiento local de la VM para habilitar permisos de ejecución
shutil.copy(drive_bin_path, local_bin_path)
os.chmod(local_bin_path, 0o755)

# Verificación rápida de ejecución del binario
proc = subprocess.run([local_bin_path], input="{}", text=True, capture_output=True)
print("✓ Binario de Haskell copiado a /tmp/validator y listo para ejecutar.")

# =============================================================================
# 3. CARGA DEL ESQUEMA FORMAL (STLC_AST_Schema.json)
# =============================================================================
schema_path = "/content/drive/MyDrive/programacion_funcional/STLC_AST_Schema.json"

if not os.path.exists(schema_path):
    # Alternativa por si quedó en la otra ruta documentada en los notebooks anteriores
    alt_path = "/content/drive/MyDrive/1i4St8f2vhpUsPZNuvMRciBeuU7uBvr2a/STLC_AST_Schema.json"
    schema_path = alt_path if os.path.exists(alt_path) else schema_path

with open(schema_path, "r", encoding="utf-8") as f:
    ast_schema = json.load(f)

print(f"✓ JSON Schema cargado exitosamente desde: {schema_path}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.8 MB/s eta 0:00:00
✓ Binario de Haskell copiado a /tmp/validator y listo para ejecutar.
✓ JSON Schema cargado exitosamente desde: /content/drive/MyDrive/programacion_funcional/STLC_AST_Schema.json


In [2]:
# =============================================================================
# CARGA DEL MODELO CUANTIZADO EN GPU (BitsAndBytes NF4)
# =============================================================================
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Liberar memoria VRAM residual previa
if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer
gc.collect()
torch.cuda.empty_cache()

# Ruta al modelo local en Drive (Qwen 2.5 7B o Llama 3.1 8B)
model_path = "/content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/qwen-2.5-7b-instruct"
# model_path = "/content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/llama-3.1-8b-instruct"

# Configuración de cuantización en 4-bit optimizada para GPU T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Cargar Tokenizer asegurando token de pad
tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Cargar Modelo CausalLM
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

print("-" * 60)
print(f"✓ Modelo cargado exitosamente: {model_path}")
print(f"✓ Dispositivo asignado: {model.device}")
print(f"✓ Ventana de contexto máxima: {getattr(model.config, 'max_position_embeddings', None)} tokens")
print("-" * 60)

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

------------------------------------------------------------
✓ Modelo cargado exitosamente: /content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/qwen-2.5-7b-instruct
✓ Dispositivo asignado: cuda:0
✓ Ventana de contexto máxima: 32768 tokens
------------------------------------------------------------


In [3]:
# =============================================================================
# DEFINICIÓN DEL PROMPT FORMAL DETERMINISTA Y PLANTILLA FEW-SHOT
# =============================================================================
schema_str = json.dumps(ast_schema, indent=2)

system_instruction = f"""Actúas como un compilador formal y determinista que traduce reglas de negocio en lenguaje natural a un Árbol de Sintaxis Abstracta (AST) en formato JSON, conforme al siguiente JSON Schema:
<JSON_SCHEMA>
{schema_str}
</JSON_SCHEMA>

REGLAS DE COMPILACIÓN OBLIGATORIAS:
1. EXHAUSTIVIDAD TOTAL: NUNCA omitas condiciones ni operaciones. Para conjunciones ("y", "and") usa "op": "&&". Para disyunciones ("o", "or") usa "op": "||".
2. TIPADO Y LITERALES:
   - `Literal`: siempre incluye "tag": "Literal", "value", y "type" con kind: "Base" ("Double", "Int", "Bool", "String").
3. IDENTIFICADORES:
   - `Var`: solo tiene "tag": "Var" y "name". NUNCA agregues la propiedad "type" a un nodo Var.
4. CONTROL DE FLUJO:
   - `IfThenElse`: contiene "tag": "IfThenElse", "cond", "then", "else".
5. PROGRAMACIÓN FUNCIONAL:
   - `Lam`: definición de función anónima con "tag": "Lam", "param", "param_type" (Type), y "body" (Expr).
   - `App`: invocación de función con "tag": "App", "func" (Expr), y "arg" (Expr). Si se invoca una función por su nombre, "func" DEBE ser un nodo "Var".
6. FORMATO DE SALIDA:
   - Devuelve ÚNICAMENTE el objeto JSON que representa la raíz del AST.
   - Sin explicaciones, comentarios ni bloques markdown (no usar ```json ni ```)."""

# Historial few-shot canónico por turnos conversacionales
PROMPT_BASE_HISTORIAL = [
    {"role": "system", "content": system_instruction},

    # Caso 1: Condicional compuesto con operadores lógicos y comparaciones
    {
        "role": "user",
        "content": "Regla de negocio:\nSi la edad es mayor o igual a 18 y el estado es igual a 'ACTIVO', devolver true; caso contrario, false."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "tag": "IfThenElse",
            "cond": {
                "tag": "BinaryOp",
                "op": "&&",
                "left": {
                    "tag": "BinaryOp",
                    "op": ">=",
                    "left": {"tag": "Var", "name": "edad"},
                    "right": {"tag": "Literal", "type": {"kind": "Base", "name": "Int"}, "value": 18}
                },
                "right": {
                    "tag": "BinaryOp",
                    "op": "==",
                    "left": {"tag": "Var", "name": "estado"},
                    "right": {"tag": "Literal", "type": {"kind": "Base", "name": "String"}, "value": "ACTIVO"}
                }
            },
            "then": {"tag": "Literal", "type": {"kind": "Base", "name": "Bool"}, "value": True},
            "else": {"tag": "Literal", "type": {"kind": "Base", "name": "Bool"}, "value": False}
        }, indent=2)
    },

    # Caso 2: Operación aritmética compuesta en ramas condicionales
    {
        "role": "user",
        "content": "Regla de negocio:\nSi la antiguedad es menor a 2 o el riesgo es igual a 'ALTO', devolver 0.0; caso contrario, devolver antiguedad * 1.5."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "tag": "IfThenElse",
            "cond": {
                "tag": "BinaryOp",
                "op": "||",
                "left": {
                    "tag": "BinaryOp",
                    "op": "<",
                    "left": {"tag": "Var", "name": "antiguedad"},
                    "right": {"tag": "Literal", "type": {"kind": "Base", "name": "Double"}, "value": 2.0}
                },
                "right": {
                    "tag": "BinaryOp",
                    "op": "==",
                    "left": {"tag": "Var", "name": "riesgo"},
                    "right": {"tag": "Literal", "type": {"kind": "Base", "name": "String"}, "value": "ALTO"}
                }
            },
            "then": {"tag": "Literal", "type": {"kind": "Base", "name": "Double"}, "value": 0.0},
            "else": {
                "tag": "BinaryOp",
                "op": "*",
                "left": {"tag": "Var", "name": "antiguedad"},
                "right": {"tag": "Literal", "type": {"kind": "Base", "name": "Double"}, "value": 1.5}
            }
        }, indent=2)
    },

    # Caso 3: Aplicación de función nombrada (App canónico con Var)
    {
        "role": "user",
        "content": "Regla de negocio:\nAplicar la función calcularImpuesto al monto 100.0."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "tag": "App",
            "func": {
                "tag": "Var",
                "name": "calcularImpuesto"
            },
            "arg": {
                "tag": "Literal",
                "type": {"kind": "Base", "name": "Double"},
                "value": 100.0
            }
        }, indent=2)
    },

    # Caso 4: Definición de función lambda (Lam)
    {
        "role": "user",
        "content": "Regla de negocio:\nDefinir una función anónima que recibe x de tipo Int y devuelve x + 1."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "tag": "Lam",
            "param": "x",
            "param_type": {"kind": "Base", "name": "Int"},
            "body": {
                "tag": "BinaryOp",
                "op": "+",
                "left": {"tag": "Var", "name": "x"},
                "right": {"tag": "Literal", "type": {"kind": "Base", "name": "Int"}, "value": 1}
            }
        }, indent=2)
    }
]

print("✓ Prompt maestro y plantilla few-shot cargados correctamente.")

✓ Prompt maestro y plantilla few-shot cargados correctamente.


In [4]:
import time
import subprocess
import json
import torch
from jsonschema import validate, ValidationError

def ejecutar_validador_haskell(ast_json: dict, env_dict: dict = None) -> dict:
    """Envía el AST y el entorno de tipos a Haskell por stdin y retorna el diagnóstico."""
    if env_dict is None:
        env_dict = {}

    payload = json.dumps({
        "ast": ast_json,
        "env": env_dict
    })

    proc = subprocess.run(
        ["/tmp/validator"],
        input=payload,
        text=True,
        capture_output=True
    )

    if proc.returncode != 0:
        return {
            "static_validation_passed": False,
            "static_error_message": f"Error del sistema Haskell: {proc.stderr.strip()}",
            "inferred_type": None
        }

    try:
        return json.loads(proc.stdout)
    except json.JSONDecodeError:
        return {
            "static_validation_passed": False,
            "static_error_message": f"Salida no válida de Haskell: {proc.stdout.strip()}",
            "inferred_type": None
        }

def compilar_y_validar(regla_negocio: str, env_dict: dict = None, max_tokens: int = 1024) -> dict:
    """Pipeline integral: NL -> LLM -> JSON Schema -> Haskell Typechecker."""
    if env_dict is None:
        env_dict = {}

    start_time = time.perf_counter()

    # 1. Construir turno conversacional dinámico
    conversacion = list(PROMPT_BASE_HISTORIAL)
    conversacion.append({
        "role": "user",
        "content": f"Regla de negocio:\n{regla_negocio}"
    })

    prompt = tokenizer.apply_chat_template(
        conversacion,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 2. Generación determinista
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # 3. Sanitización de Markdown
    ast_clean = raw_output
    if ast_clean.startswith("```json"):
        ast_clean = ast_clean[7:]
    if ast_clean.startswith("```"):
        ast_clean = ast_clean[3:]
    if ast_clean.endswith("```"):
        ast_clean = ast_clean[:-3]
    ast_clean = ast_clean.strip()

    # Registro de estado
    registro = {
        "prompt_input": regla_negocio,
        "raw_output": raw_output,
        "syntax_valid": False,
        "static_validation_passed": False,
        "static_error_message": None,
        "inferred_type": None,
        "ast_json": None,
        "execution_time_ms": 0.0
    }

    # 4. Validación Sintáctica (JSON + JSON Schema)
    try:
        ast_json = json.loads(ast_clean)
        validate(instance=ast_json, schema=ast_schema)
        registro["syntax_valid"] = True
        registro["ast_json"] = ast_json
    except json.JSONDecodeError as e:
        registro["static_error_message"] = f"JSON malformado: {str(e)}"
        registro["execution_time_ms"] = round((time.perf_counter() - start_time) * 1000, 2)
        return registro
    except ValidationError as e:
        registro["static_error_message"] = f"Violación de JSON Schema: {e.message}"
        registro["execution_time_ms"] = round((time.perf_counter() - start_time) * 1000, 2)
        return registro

    # 5. Validación Semántica Formal en Haskell
    diag_haskell = ejecutar_validador_haskell(ast_json, env_dict)

    registro["static_validation_passed"] = diag_haskell.get("static_validation_passed", False)
    registro["static_error_message"] = diag_haskell.get("static_error_message")
    registro["inferred_type"] = diag_haskell.get("inferred_type")

    registro["execution_time_ms"] = round((time.perf_counter() - start_time) * 1000, 2)
    return registro

print("✓ Función de integración 'compilar_y_validar' definida correctamente.")

✓ Función de integración 'compilar_y_validar' definida correctamente.


## **Prueba**

¿Qué función cumple en esta demostración los 'env'?

 Cumple la función del "árbitro" o auditor estático.   Si no le pasamos el entorno al validador en Haskell, el cálculo lambda puro solo admitiría expresiones cerradas (funciones donde todas las variables se definen dentro de la misma regla mediante lambdas, como $\lambda x . x + 1$).   

 Pero las reglas de negocio reales son expresiones con variables libres que hacen referencia a datos externos (al cliente, a la factura, al empleado).   

 En la demostración, incluir env permite probar dos cosas críticas para tu experimento:

 1. Que el validador es capaz de verificar la consistencia con el negocio:Si el LLM traduce "edad mayor o igual a 18" como edad >= 18, Haskell consulta env, ve que edad existe en el sistema y es un entero, y declara que la regla es segura y válida.   
 2. Demostrar cómo frena las alucinaciones del LLM:
Imaginá que el usuario escribe "Si el cliente es confiable..." y el LLM alucina un campo inventado que no existe en tu base de datos: score_crediticio > 50.
Como score_crediticio no está en env, Haskell bloquea inmediatamente la regla con un error:   "Error de Scope: Variable libre no declarada 'score_crediticio'"   

Sin env, un lenguaje dinámico como Python ejecutaría ese código y recién explotaría en runtime cuando un cliente real intente hacer una operación (NameError: name 'score_crediticio' is not defined). Haskell lo frena antes de tocar producción.   

En resumen: en el experimento simulamos que tu sistema ya tiene registradas las variables del negocio (env), el usuario solo introduce el texto en lenguaje natural, y nosotros evaluamos si el LLM genera código que respeta ese contexto y si Haskell es capaz de auditarlo exitosamente

In [5]:
# =============================================================================
# BATERÍA DE PRUEBAS: INTEGRACIÓN LLM + VALIDADOR FORMAL HASKELL
# =============================================================================

casos_benchmark = [
    # 1. Regla condicional compuesta con variables de negocio
    {
        "id": "RULE_001",
        "descripcion": "Condicional anidado sobre clientes",
        "enunciado": "Si la edad es mayor o igual a 18 y el estado es igual a 'ACTIVO', evaluar si el balance es mayor a 0.0 o tiene_garantia es true; si se cumple, devolver 1.0; caso contrario, devolver 0.0.",
        "env": {
            "edad": {"kind": "Base", "name": "Int"},
            "estado": {"kind": "Base", "name": "String"},
            "balance": {"kind": "Base", "name": "Double"},
            "tiene_garantia": {"kind": "Base", "name": "Bool"}
        }
    },

    # 2. Regla aritmética con precedencia y multiplicadores
    {
        "id": "RULE_002",
        "descripcion": "Cálculo salarial con bonificación/deducción",
        "enunciado": "Si el salario es menor o igual a 50000.0, devolver (salario * 1.20) + 500.0; caso contrario, devolver salario - (salario * 0.05).",
        "env": {
            "salario": {"kind": "Base", "name": "Double"}
        }
    },

    # 3. Abstracción Lambda local
    {
        "id": "RULE_003",
        "descripcion": "Definición de función anónima con condicional",
        "enunciado": "Definir una función anónima que recibe una variable puntaje de tipo Int y hace lo siguiente: si puntaje es mayor a 70, devuelve true; caso contrario, devuelve false.",
        "env": {}  # No requiere variables libres externas
    },

    # 4. Invocación de función nombrada existente en el entorno
    {
        "id": "RULE_004",
        "descripcion": "Aplicación de función predefinida",
        "enunciado": "Aplicar la función calcularImpuesto al monto 100.0.",
        "env": {
            "calcularImpuesto": {
                "kind": "Arrow",
                "from": {"kind": "Base", "name": "Double"},
                "to": {"kind": "Base", "name": "Double"}
            },
            "monto": {"kind": "Base", "name": "Double"}
        }
    },

    # 5. Función de orden superior currificada
    {
        "id": "RULE_005",
        "descripcion": "Currificación (Lam -> Lam)",
        "enunciado": "Definir una función que recibe una base de tipo Double y retorna otra función que recibe un exponente de tipo Double y devuelve base * exponente.",
        "env": {}
    }
]

print("=" * 80)
print("INICIANDO BENCHMARK INTEGRADO: LLM -> AST -> GUARDRAIL HASKELL")
print("=" * 80)

resultados_sesion = []

for idx, caso in enumerate(casos_benchmark, 1):
    print(f"\n{'#' * 80}")
    print(f"▶ PRUEBA {idx}/{len(casos_benchmark)} | ID: {caso['id']} - {caso['descripcion']}")
    print(f"{'#' * 80}")

    # 1. Entrada
    print("\n[1] INPUT (REGLA DE NEGOCIO):")
    print(f"\"{caso['enunciado']}\"")
    print(f"Entorno Gamma provisto: {list(caso['env'].keys()) if caso['env'] else 'Vacío (Variables cerradas)'}")

    # Ejecutar pipeline
    resultado = compilar_y_validar(caso["enunciado"], caso["env"])
    resultados_sesion.append((caso["id"], resultado))

    # 2. AST generado por el LLM
    print("\n[2] AST GENERADO POR EL LLM:")
    if resultado["ast_json"]:
        print(json.dumps(resultado["ast_json"], indent=2))
    else:
        print("Raw Output (No formateable como JSON válido):")
        print(resultado["raw_output"])

    # 3. Accionar del validador formal
    print("\n[3] ACCIONAR DEL VALIDADOR FORMAL EN HASKELL:")
    print(f"- Validez sintáctica (JSON Schema) : {'✓ VÁLIDO' if resultado['syntax_valid'] else '✗ INVÁLIDO'}")

    if resultado["static_validation_passed"]:
        print(f"- Validación formal (Haskell)      : ✓ APROBADO")
        print(f"- Tipo inferido deducido           : {resultado['inferred_type']}")
    else:
        print(f"- Validación formal (Haskell)      : ✗ BLOQUEADO")
        print(f"- Motivo del rechazo formal        : {resultado['static_error_message']}")

    print(f"- Tiempo de procesamiento total    : {resultado['execution_time_ms']} ms")

# Resumen ejecutivo final
print("\n" + "=" * 80)
print("RESUMEN GENERAL DEL BENCHMARK")
print("=" * 80)
for cid, r in resultados_sesion:
    estado_haskell = "APROBADO" if r["static_validation_passed"] else "BLOQUEADO"
    print(f"[{cid}] Schema: {'OK' if r['syntax_valid'] else 'FAIL'} | Haskell: {estado_haskell:9} | Tipo: {str(r['inferred_type']):25} | Tiempo: {r['execution_time_ms']} ms")
print("=" * 80)

INICIANDO BENCHMARK INTEGRADO: LLM -> AST -> GUARDRAIL HASKELL

################################################################################
▶ PRUEBA 1/5 | ID: RULE_001 - Condicional anidado sobre clientes
################################################################################

[1] INPUT (REGLA DE NEGOCIO):
"Si la edad es mayor o igual a 18 y el estado es igual a 'ACTIVO', evaluar si el balance es mayor a 0.0 o tiene_garantia es true; si se cumple, devolver 1.0; caso contrario, devolver 0.0."
Entorno Gamma provisto: ['edad', 'estado', 'balance', 'tiene_garantia']

[2] AST GENERADO POR EL LLM:
{
  "tag": "IfThenElse",
  "cond": {
    "tag": "BinaryOp",
    "op": "&&",
    "left": {
      "tag": "BinaryOp",
      "op": ">=",
      "left": {
        "tag": "Var",
        "name": "edad"
      },
      "right": {
        "tag": "Literal",
        "type": {
          "kind": "Base",
          "name": "Int"
        },
        "value": 18
      }
    },
    "right": {
      "tag":

## **Pruebas mas amplias y rigurosas**

In [6]:
import os
import json
import time
import subprocess
from datetime import datetime, timezone
from jsonschema import validate, ValidationError

# -----------------------------------------------------------------------------
# 1. Configuración de persistencia según OutputRecordSchema
# -----------------------------------------------------------------------------
output_jsonl_path = "/content/drive/MyDrive/programacion_funcional/experimentos.jsonl"
os.makedirs(os.path.dirname(output_jsonl_path), exist_ok=True)

# -----------------------------------------------------------------------------
# 2. Corpus de Prueba (Casos canónicos, estrés de scope y estrés de tipos)
# -----------------------------------------------------------------------------
corpus_evaluacion = [
    # Caso 1: Canónico válido
    {
        "case_id": "RULE_001_VALID",
        "prompt_input": "Si la antiguedad es mayor a 5, devolver el salario multiplicado por 1.10; caso contrario devolver el salario.",
        "env": {
            "antiguedad": {"kind": "Base", "name": "Int"},
            "salario": {"kind": "Base", "name": "Double"}
        },
        "test_input": {"antiguedad": 6, "salario": 1000.0},
        "expected_output": 1100.0
    },
    # Caso 2: Riesgo de Scope (variable no provista en el entorno formal)
    {
        "case_id": "RULE_002_SCOPE_STRESS",
        "prompt_input": "Si el cliente tiene categoria VIP y el balance es mayor a 500.0, devolver 0.0; caso contrario devolver balance * 0.02.",
        "env": {
            # Intencionalmente no se incluye 'categoria' para auditar alucinación de scope
            "balance": {"kind": "Base", "name": "Double"}
        },
        "test_input": {"balance": 1000.0},
        "expected_output": 20.0
    },
    # Caso 3: Riesgo de Tipo (ramas heterogéneas: texto vs número)
    {
        "case_id": "RULE_003_TYPE_STRESS",
        "prompt_input": "Si la edad es menor a 18, devolver 'MENOR'; caso contrario devolver 100.0.",
        "env": {
            "edad": {"kind": "Base", "name": "Int"}
        },
        "test_input": {"edad": 25},
        "expected_output": 100.0
    }
]

# -----------------------------------------------------------------------------
# 3. Función auxiliar para guardar registros en experimentos.jsonl
# -----------------------------------------------------------------------------
def registrar_experimento(record_dict):
    record_dict["timestamp"] = datetime.now(timezone.utc).isoformat()
    # Guardar en archivo JSON Lines
    with open(output_jsonl_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record_dict, ensure_ascii=False) + "\n")

# -----------------------------------------------------------------------------
# 4. Evaluador del Carril Propuesto (DSL AST + Haskell)
# -----------------------------------------------------------------------------
def evaluar_carril_propuesto(caso):
    t_start = time.perf_counter()

    # 1. Generación e inferencia con el prompt AST existente
    resultado = compilar_y_validar(caso["prompt_input"], caso["env"])

    t_end = time.perf_counter()
    duration_ms = round((t_end - t_start) * 1000, 2)

    # Registro de auditoría conforme a OutputRecordSchema
    record = {
        "case_id": caso["case_id"],
        "lane": "proposed_dsl_haskell",
        "prompt_input": caso["prompt_input"],
        "raw_output": resultado["raw_output"],
        "syntax_valid": resultado["syntax_valid"],
        "static_validation_passed": resultado["static_validation_passed"],
        "static_error_message": resultado["static_error_message"],
        "runtime_passed": False,         # Se actualiza con el evaluador
        "runtime_error_message": None,
        "test_passed": False,
        "execution_time_ms": duration_ms
    }

    registrar_experimento(record)
    return record

print("✓ Harness de evaluación configurado. Listo para ejecutar el pipeline sobre el corpus.")

✓ Harness de evaluación configurado. Listo para ejecutar el pipeline sobre el corpus.


In [7]:
import json

resultados_carril_propuesto = []

print("=" * 80)
print("INICIANDO EVALUACIÓN DEL CARRIL PROPUESTO (DSL AST + HASKELL)")
print("=" * 80)

for idx, caso in enumerate(corpus_evaluacion, 1):
    print(f"\n[{idx}/{len(corpus_evaluacion)}] Procesando: {caso['case_id']}")
    print(f"Regla NL: \"{caso['prompt_input']}\"")
    print(f"Entorno Gamma provisto: {list(caso['env'].keys())}")

    # Ejecutar compilación con LLM, validación formal y persistencia en JSONL
    record = evaluar_carril_propuesto(caso)
    resultados_carril_propuesto.append(record)

    print("\n--- Resultado de la auditoría ---")
    print(f"Sintaxis JSON válida          : {record['syntax_valid']}")
    print(f"Validación formal (Haskell)   : {record['static_validation_passed']}")
    if record['static_error_message']:
        print(f"Error estático interceptado   : {record['static_error_message']}")
    print(f"Tiempo total                  : {record['execution_time_ms']} ms")
    print("-" * 80)

print("\n✓ Todos los casos fueron procesados y persistidos en experimentos.jsonl")

INICIANDO EVALUACIÓN DEL CARRIL PROPUESTO (DSL AST + HASKELL)

[1/3] Procesando: RULE_001_VALID
Regla NL: "Si la antiguedad es mayor a 5, devolver el salario multiplicado por 1.10; caso contrario devolver el salario."
Entorno Gamma provisto: ['antiguedad', 'salario']

--- Resultado de la auditoría ---
Sintaxis JSON válida          : True
Validación formal (Haskell)   : True
Tiempo total                  : 20971.03 ms
--------------------------------------------------------------------------------

[2/3] Procesando: RULE_002_SCOPE_STRESS
Regla NL: "Si el cliente tiene categoria VIP y el balance es mayor a 500.0, devolver 0.0; caso contrario devolver balance * 0.02."
Entorno Gamma provisto: ['balance']

--- Resultado de la auditoría ---
Sintaxis JSON válida          : True
Validación formal (Haskell)   : False
Error estático interceptado   : Error de Scope: Variable libre no declarada 'categoria'
Tiempo total                  : 31302.21 ms
------------------------------------------------

**1) ¿Qué se busca en esta prueba?**

El objetivo metodológico de este experimento es evaluar la efectividad de una capa de verificación formal como mecanismo de defensa (guardrail) frente a la naturaleza probabilística de los LLM cuando generan lógica para procesos determinísticos.   

Específicamente, la prueba busca responder a la pregunta de investigación de tu tesis: comprobar si restringir la respuesta del modelo a un Árbol de Sintaxis Abstracta (AST) y someterlo a comprobación estática de tipos (typechecking) y alcance (scope checking) en Haskell permite interceptar alucinaciones e inconsistencias lógicas antes de que se manifiesten como errores de ejecución (runtime errors).   

Para demostrarlo, se diseñó un corpus con tres escenarios intencionales:   Control canónico (RULE_001_VALID):
1. Una regla válida donde el LLM debe acertar y el sistema debe dar luz verde.   
2. Estrés de alcance / Scope (RULE_002_SCOPE_STRESS): Una regla con un concepto de negocio ausente en el entorno del sistema para comprobar si el LLM alucina variables no registradas.   
3. Estrés de tipos (RULE_003_TYPE_STRESS): Una regla con ramas heterogéneas para evaluar si el sistema permite inconsistencias que violen la homogeneidad matemática del cálculo lambda.   

**2) Resultados Obtenidos en la Ejecución**

Los registros persistidos en experimentos.jsonl arrojaron métricas claras:   

*En RULE_001_VALID (Caso Correcto):*

*El LLM construyó un AST impecable.   
*Validación sintáctica (JSON Schema): syntax_valid = true.   
*Validación formal en Haskell: static_validation_passed = true (sin mensajes de error).   
*Comportamiento: Aprobación legítima; se autoriza la evaluación.   

*En RULE_002_SCOPE_STRESS (Alucinación de Variable):*

*El LLM asumió e inventó la variable categoria (creó un nodo {"tag": "Var", "name": "categoria"}).   
*Validación sintáctica (JSON Schema): syntax_valid = true.   
*Validación formal en Haskell: static_validation_passed = false.   
*Error interceptado: "Error de Scope: Variable libre no declarada 'categoria'".   
*Comportamiento: Bloqueo preventivo de la alucinación.   

*En RULE_003_TYPE_STRESS (Discrepancia de Tipos en Ramas):*

*El LLM generó un condicional devolviendo un Literal String ("MENOR") en la rama then y un Literal Double (100.0) en la rama else.   

*Validación sintáctica (JSON Schema): syntax_valid = true.   Validación formal en Haskell: static_validation_passed = false.   

*Error interceptado: "Discrepancia de Tipos en ramas Then y Else de IfThenElse: se esperaba Base TString pero se obtuvo Base TDouble".   

*Comportamiento: Bloqueo preventivo por violación formal del sistema de tipos.  

**3) ¿Dónde está el valor agregado del accionar de Haskell aquí?**

El valor agregado se hace evidente al contrastar las tres capas del problema:

*La insuficiencia del JSON Schema:*

En los tres casos analizados, la salida del LLM obtuvo syntax_valid: true. Para un esquema JSON estándar, Var solo necesita una propiedad "name" de tipo string, y las ramas de IfThenElse solo necesitan ser expresiones JSON válidas. El esquema es completamente ciego a la semántica del negocio y al contexto léxico. Por sí solo, no puede evitar que una alucinación pase a producción.   

*Lo que habría ocurrido en un lenguaje imperativo (Python libre):*

Si este código se hubiera emitido en Python sin verificación (Baseline 1):   

*El caso 2 habría intentado leer la variable categoria, lanzando un fallo fatal en tiempo de ejecución: NameError: name 'categoria' is not defined.   

*El caso 3 habría devuelto dinámicamente un string o un float de forma impredecible, rompiendo cálculos posteriores con excepciones de tipo (TypeError) en operaciones que asumían un resultado numérico homogéneo.   

*El verdadero aporte del validador en Haskell (Neuro-simbólico):*

Haskell actúa como un compilador matemático que audita el AST antes de que toque la memoria de ejecución:   

*Auditoría de Entorno ($\Gamma$): Garantiza que ninguna variable libre fuera del catálogo del sistema sea evaluada.   

*Garantía de Preservación y Tipado Homogéneo: Aplica las reglas del cálculo lambda simplemente tipado (STLC), asegurando que todas las ramas condicionales converjan a un tipo determinístico idéntico.   

*Migración del fallo: Convierte una excepción en runtime (que tira un servicio en producción) en un rechazo estático estructurado, permitiendo al sistema notificar el error de forma segura o pedirle al LLM que regenere la respuesta de manera asistida.   

## **Mas pruebas sobre el validador**

In [8]:
corpus_cierre_haskell = [
    # 1. Operación binaria cruzada (Double vs String en comparación)
    {
        "case_id": "RULE_004_OP_TYPE_MISMATCH",
        "prompt_input": "Si el balance es mayor a 'ALTO', devolver 0.0; caso contrario devolver balance.",
        "env": {
            "balance": {"kind": "Base", "name": "Double"}
        }
    },
    # 2. Coerción implícita rechazada (condición no booleana)
    {
        "case_id": "RULE_005_NON_BOOL_COND",
        "prompt_input": "Si el salario, devolver salario * 1.10; caso contrario devolver 0.0.",
        "env": {
            "salario": {"kind": "Base", "name": "Double"}
        }
    },
    # 3. Aplicación inválida sobre tipo escalar (App sobre no-función)
    {
        "case_id": "RULE_006_APP_NON_FUNCTION",
        "prompt_input": "Aplicar la variable balance al valor 50.",
        "env": {
            "balance": {"kind": "Base", "name": "Double"}
        }
    }
]

print("=" * 80)
print("INICIANDO BATERÍA DE CIERRE FORMAL EN HASKELL")
print("=" * 80)

for caso in corpus_cierre_haskell:
    print(f"\nProcesando: {caso['case_id']}")
    print(f"Regla NL: \"{caso['prompt_input']}\"")

    record = evaluar_carril_propuesto(caso)

    print(f"Sintaxis JSON válida       : {record['syntax_valid']}")
    print(f"Validación formal (Haskell): {record['static_validation_passed']}")
    print(f"Mensaje interceptado       : {record['static_error_message']}")
    print("-" * 80)

INICIANDO BATERÍA DE CIERRE FORMAL EN HASKELL

Procesando: RULE_004_OP_TYPE_MISMATCH
Regla NL: "Si el balance es mayor a 'ALTO', devolver 0.0; caso contrario devolver balance."
Sintaxis JSON válida       : True
Validación formal (Haskell): False
Mensaje interceptado       : Operación Binaria Inválida (Gt): tipos incompatibles left=Base TDouble, right=Base TString
--------------------------------------------------------------------------------

Procesando: RULE_005_NON_BOOL_COND
Regla NL: "Si el salario, devolver salario * 1.10; caso contrario devolver 0.0."
Sintaxis JSON válida       : True
Validación formal (Haskell): False
Mensaje interceptado       : Discrepancia de Tipos en condición de IfThenElse: se esperaba Base TBool pero se obtuvo Base TDouble
--------------------------------------------------------------------------------

Procesando: RULE_006_APP_NON_FUNCTION
Regla NL: "Aplicar la variable balance al valor 50."
Sintaxis JSON válida       : True
Validación formal (Haskell): F

Esta segunda tanda de evaluaciones se enfocó en estresar las reglas de tipado complejas del Cálculo Lambda Simplemente Tipado ($\lambda^{\to}$ o STLC) que van más allá del alcance de variables y de la homogeneidad básica de ramas:   

RULE_004_OP_TYPE_MISMATCH: Evaluar la coherencia de tipos en operadores binarios relacionales (BinaryOp).   

RULE_005_NON_BOOL_COND: Evaluar el rechazo estricto a coerciones implícitas (truthy/falsy) en expresiones condicionales (IfThenElse).   

RULE_006_APP_NON_FUNCTION: Evaluar la preservación de la semántica funcional en la aplicación de términos (App).   

**Caso 4: RULE_004_OP_TYPE_MISMATCH (Comparación Relacional Heterogénea)**

*Entrada NL: "Si el balance es mayor a 'ALTO', devolver 0.0; caso contrario devolver balance."   

*Comportamiento del LLM: Generó un AST sintácticamente válido donde construyó un nodo BinaryOp con operador > comparando la variable balance contra el literal de texto "ALTO" (Base TString).   

*Ceguera del JSON Schema: El validador sintáctico dio luz verde (syntax_valid: true) porque la gramática JSON de BinaryOp solo exige que left y right sean nodos de tipo Expr válidos, sin capacidad de inspeccionar sus tipos subyacentes.   

*Intercepción en Haskell: Al evaluar los tipos en el entorno $\Gamma$, el typechecker detectó que balance es Base TDouble y el literal es Base TString. Como los operadores relacionales de orden sólo están tipados para operandos numéricos homogéneos, bloqueó la regla:   static_error_message: "Operación Binaria Inválida (Gt): tipos incompatibles left=Base TDouble, right=Base TString"   

*Valor agregado: En un entorno imperativo dinámico como Python, esta comparación habría superado la fase de análisis y arrojado una excepción crítica en tiempo de ejecución: TypeError: '>' not supported between instances of 'float' and 'str'.

**Caso 5: RULE_005_NON_BOOL_COND (Condición Escalar / Coerción Falsy Rechazada)**

*Entrada NL: "Si el salario, devolver salario * 1.10; caso contrario devolver 0.0."   

*Comportamiento del LLM: Ante una redacción informal y elíptica en lenguaje natural, el modelo ubicó directamente el identificador salario dentro del campo cond del constructor IfThenElse.   

*Ceguera del JSON Schema: Aprobó la estructura (syntax_valid: true), dado que Var es una expresión formalmente admitida dentro de cond.   

*Intercepción en Haskell: El typechecker aplicó la regla formal del cálculo lambda para bifurcaciones, la cual exige de manera axiomática que la condición sea estrictamente de tipo booleano ($\text{cond} : \text{Bool}$). Al constatar que salario tiene tipo Base TDouble, rechazó el AST:   static_error_message: "Discrepancia de Tipos en condición de IfThenElse: se esperaba Base TBool pero se obtuvo Base TDouble"   

*Valor agregado: En la mayoría de los lenguajes dinámicos (Python, JavaScript), expresiones como if salario: son evaluadas por coerción booleana implícita (falsy si es 0.0, truthy en cualquier otro valor). Haskell impone disciplina matemática estricta y evita interpretaciones ambiguas en reglas de negocio críticas.

**Caso 6: RULE_006_APP_NON_FUNCTION (Invocación sobre Tipo Escalar)**

*Entrada NL: "Aplicar la variable balance al valor 50."   

*Comportamiento del LLM: Interpretó la orden de invocación creando un nodo App, asignando la variable balance al campo func y el literal 50 (Base TInt) al campo arg.   

*Ceguera del JSON Schema: El esquema lo catalogó como válido (syntax_valid: true), pues App únicamente requiere dos subárboles Expr (func y arg) sin evaluar la naturaleza de primer o segundo orden de los mismos.   

*Intercepción en Haskell: Al auditar la aplicación pura ($e_1 \; e_2$), el validador verificó que $e_1$ posea una signatura funcional flecha ($T_1 \to T_2$). Al comprobar en $\Gamma$ que balance es un dato escalar primitivo (Base TDouble), detuvo inmediatamente la operación:   static_error_message: "Error de Aplicación en posición de función en App: se intentó aplicar un término de tipo no funcional Base TDouble"   

*Valor agregado: Convirtió un error fatal de runtime (equivalente a intentar invocar un número como rutina: TypeError: 'float' object is not callable en Python) en una intercepción estática explicable y trazable antes de la ejecución.   